# [CodeReview]nano vllm

## 仓库结构
```
nanovllm/
├── __init__.py          # 暴露 LLM, SamplingParams
├── config.py            # Config dataclass（模型路径、KV cache 参数等）
├── llm.py               # LLM 类 = LLMEngine 的别名
├── sampling_params.py   # SamplingParams dataclass
├── engine/
│   ├── llm_engine.py    # 主引擎：generate/step/add_request 入口
│   ├── scheduler.py     # 调度器：prefill / decode / preempt
│   ├── sequence.py      # Sequence（序列状态机 WAITING→RUNNING→FINISHED）
│   ├── block_manager.py # 显存 block 管理 + Prefix Cache（xxhash 实现）
│   └── model_runner.py  # 模型执行器：warmup / KV cache 分配 / CUDA Graph / TP
├── layers/
│   ├── activation.py    # SiluAndMul
│   ├── attention.py     # FlashAttention 调用 + Triton KV cache 存储
│   ├── embed_head.py    # VocabParallelEmbedding + ParallelLMHead（TP 感知）
│   ├── layernorm.py     # RMSNorm（含 fused add-rms）
│   ├── linear.py        # Column/Row/QKV/Merged parallel linear（TP 拆分）
│   ├── rotary_embedding.py # RoPE + @torch.compile
│   └── sampler.py       # 采样（temperature + 随机）
├── models/
│   └── qwen3.py         # Qwen3ForCausalLM（Qwen3Attention/MLP/DecoderLayer/Model）
└── utils/
    ├── context.py       # Thread-local 上下文（prefill/decode 参数传递）
    └── loader.py        # safetensors 权重加载 + weight_loader 分发
```

确保cuda toolkit等正常安装